# Lab 2: Messages & Tools

**Difficulty: Beginner | ~30 min | Requires Lab 1 (recommended)**

Messages and Tools are the two building blocks every agent is made of. Messages are the data agents pass around; Tools are the actions they take. You will create the four kinds of messages by hand, send a whole conversation to a model, write two tools, see exactly what the model sees, and then watch an agent's conversation happen as a list of messages.


## Step 1

One command installs all required modules, versions pinned so the lab is reproducible.

In [ ]:
# One command installs all required modules (versions pinned for reproducibility)
!pip install -qU "langchain==1.2.15" "langchain-core==1.2.28" "langchain-openai==1.1.12" "python-dotenv==1.2.2"


## Step 2

Loads your OpenRouter API key from `.env` and stops with a clear message if it's missing.

In [ ]:
import os
from dotenv import load_dotenv

# Read the OPENROUTER_API_KEY we saved in .env (Section 9 of the guide)
load_dotenv()

# Stop early with a clear message if the key is missing
if not os.getenv("OPENROUTER_API_KEY"):
    raise SystemExit("No OPENROUTER_API_KEY found. Add it to .env and restart the kernel.")

## Step 3

Create the four kinds of messages by hand — a message is just an object with a `type` and `content`.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# Each message type plays one role in the conversation
system_message = SystemMessage(content="You are a math helper.")   # the rules
human_message = HumanMessage(content="What is 8 + 7?")             # your words
ai_message = AIMessage(content="I'll compute that with a tool.")   # the model's words
tool_message = ToolMessage(content="15", tool_call_id="call_1")    # a tool's result

# Every message has a .type and a .content — the two things that matter
for message in [system_message, human_message, ai_message, tool_message]:
    print(f"{message.type:<8} -> {message.content}")

## Step 4

A conversation is just a list of messages — the model reads it top to bottom.

In [ ]:
# A chat is just a list of messages, in order, oldest first
history = [
    SystemMessage(content="You are a friendly assistant."),
    HumanMessage(content="Name three colors of a traffic light."),
    AIMessage(content="Red, yellow, and green."),
    HumanMessage(content="Which one means stop?"),
]

# Print the list so you can see the whole conversation at once
for message in history:
    print(f"{message.type:<8} -> {message.content}")

## Step 5

Send the whole history to a chat model and watch it answer the last question.

In [ ]:
from langchain_openai import ChatOpenAI

# The model: same wrapper and settings as Lab 1
model = ChatOpenAI(
    model="openai/gpt-oss-20b:free",         # a free model on OpenRouter
    base_url="https://openrouter.ai/api/v1", # redirect the OpenAI client to OpenRouter
    api_key=os.getenv("OPENROUTER_API_KEY"), # your key, read from .env
    temperature=0,                           # 0 = factual, reproducible
)

# The model reads the whole history and answers the last question
reply = model.invoke(history)
print(f"reply type: {reply.type}")
print(reply.content)

## Step 6

A tool is a plain Python function with a docstring and type hints.

In [ ]:
# A tool is just a function. The docstring tells the model WHAT it does,
# the type hints tell it WHAT INPUTS to supply.
def add(a: float, b: float) -> float:
    """Add two numbers and return their sum."""
    return a + b

def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return their product."""
    return a * b

## Step 7

The model never sees your Python code — it sees a JSON schema built from the docstring and type hints.

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

# Build the schema for each tool and print what the model actually sees
for tool_function in [add, multiply]:
    schema = convert_to_openai_tool(tool_function)["function"]
    print(schema["name"])        # the tool's name
    print(schema["description"]) # from the docstring
    print(schema["parameters"])  # from the type hints
    print()

## Step 8

Wrap the model and both tools into the agent loop.

In [ ]:
from langchain.agents import create_agent

# The agent = this model + both tools + the decision loop around them
agent = create_agent(model, tools=[add, multiply])

## Step 9

Run the agent and read the whole conversation as a list of messages — watch the tool round-trip.

In [ ]:
# Run the agent loop with one user message
result = agent.invoke({
    "messages": [("user", "What is 8 multiplied by 7?")],
})

# Print the whole conversation, message by message
for message in result["messages"]:
    print(f"\n--- {message.type} ---")                        # which kind of message
    print(message.content if message.content else "(no text)") # its text
    for call in getattr(message, "tool_calls", []):            # did the model ask for a tool?
        print(f"  tool call: {call['name']}{call['args']}")

## Step 10

A question that needs both tools — the loop repeats until the model can answer.

In [ ]:
# One question that needs both tools
result_both = agent.invoke({
    "messages": [("user", "What is 8 multiplied by 7? And what is 5 plus 3?")],
})

# The message types alone tell the story of the loop
for message in result_both["messages"]:
    print(message.type)

# The last message is always the final answer
print(result_both["messages"][-1].content)

## Optional Exercise

Add a third tool. In a new cell, write a `subtract` function (docstring: "Subtract the second number from the first and return the result.", inputs `a: float, b: float`), rebuild the agent with `tools=[add, multiply, subtract]`, and ask it "What is 20 minus 8?". Confirm the message trail ends `human -> ai -> tool -> ai` and the answer is 12.